# TV3 — Train trên Kaggle (UCF-Crime)
Notebook này chạy: cài đặt → index dữ liệu → test → **(A) MIL baseline trên feature R(2+1)D K400** (task tuần 2) →
**(B) fine-tune M3 Video Swin-T** (task tuần 3) → trích feature Swin + MIL → đánh giá test.

**Trước khi chạy:** Settings → Accelerator **GPU (T4 x2 hoặc P100)**, **Internet: On**; *Add Input* → dataset
`odins0n/ucf-crime-dataset`. Xem đầy đủ trong `docs/kaggle.md`.

## 0. Cài đặt

In [ ]:
REPO = "https://github.com/quocnv20214926/Deeplearning_project.git"
BRANCH = "feat/tv3-train-framework"   # đổi thành master sau khi PR được merge
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /kaggle/working
!rm -rf /kaggle/working/Deeplearning_project && git clone -q -b $BRANCH $REPO /kaggle/working/Deeplearning_project
%cd /kaggle/working/Deeplearning_project
!git log --oneline -3
!pip install -q pytorchvideo

In [ ]:
# Tìm thư mục dataset (chứa Train/ và Test/) trong /kaggle/input
import os, glob
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/Train", recursive=True)
         if os.path.isdir(p.replace("Train", "Test"))]
assert cands, "Không thấy dataset: Add Input -> odins0n/ucf-crime-dataset"
FRAMES_ROOT = cands[0]
print("frames_root =", FRAMES_ROOT); print(os.listdir(FRAMES_ROOT))
print(sorted(os.listdir(f"{FRAMES_ROOT}/Train"))[:5], os.listdir(f"{FRAMES_ROOT}/Train/Abuse")[:3])

# Mọi script (train, evaluate, prepare_data...) đều đọc CFG_OVERRIDES -> không phải sửa base.yaml
os.environ["CFG_OVERRIDES"] = f"data.frames_root={FRAMES_ROOT};train.num_workers=4"

In [ ]:
# (tuỳ chọn) wandb: Add-ons -> Secrets -> thêm WANDB_API_KEY. Không có thì bỏ qua, log vẫn ghi CSV.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    WANDB = ""
except Exception:
    os.environ["WANDB_MODE"] = "disabled"; WANDB = "--no_wandb"
print("wandb:", "bật" if not WANDB else "tắt")

## 1. Index dữ liệu (~5–10 phút) + test
Split train/val/test đã có sẵn trong git (`data/processed/splits/`) — **không** chạy lại bước `splits`.

In [ ]:
!python scripts/prepare_data.py --steps index
!ls -lh data/processed/
!python -m pytest -q tests/ 2>&1 | tail -3

## 2. Chạy thử (2 video/lớp, 1 epoch) — phải chạy được trước khi chạy thật

In [ ]:
!python -m src.train --config configs/m1_r2plus1d.yaml --limit_videos 2 --epochs 1 --no_wandb --tag smoke
!python -m src.train --config configs/m3_swin3d.yaml --limit_videos 2 --epochs 1 --no_wandb --tag smoke_swin
!rm -f weights/smoke* results/logs/smoke*

## A. Task tuần 2 — MIL baseline trên feature R(2+1)D pretrained K400 (~20–30 phút)
Kiểm tra cả pipeline MIL khi M1 của TV1 chưa fine-tune xong. Feature để riêng ở `data/features_k400/` để không lẫn
với feature từ checkpoint M1 sau này. Kỳ vọng frame AUC test khoảng 0,70–0,80; ~0,5 là có lỗi.

In [ ]:
!python -m src.extract_features --config configs/m1_r2plus1d.yaml --out_root data/features_k400 --batch_size 128
!python -m src.train_mil --config configs/m1_r2plus1d.yaml --features data/features_k400/r2plus1d_18 --tag r2plus1d_18_k400

## B. Task tuần 3 — Fine-tune M3 Video Swin-T (cls)
~40k clip/epoch, tối đa 20 epoch, early stopping patience 5. Chạy ô dưới để đo tốc độ/VRAM ở vài trăm bước đầu,
sau đó chạy thật bằng **Save Version → Save & Run All (Commit)** để chạy nền tối đa 12 giờ (tắt trình duyệt được).
Nếu OOM: thêm `--set train.batch_size=4 train.grad_accum=4` hoặc `model.grad_checkpointing=true`.

In [ ]:
# đo nhanh: 1 epoch rút gọn 200 bước optimizer
!python -m src.train --config configs/m3_swin3d.yaml --epochs 1 --max_steps 200 --no_wandb --tag swin_speedtest
!rm -f weights/swin_speedtest* 

In [ ]:
# Chạy thật. Bị ngắt giữa chừng: tạo version mới, Add Input = output version cũ, rồi bỏ comment 2 dòng dưới
# !cp /kaggle/input/<ten-output-cu>/Deeplearning_project/weights/swin3d_t_cls*.pt weights/
# RESUME = "--resume weights/swin3d_t_cls_last.pt"
RESUME = ""
!python -m src.train --config configs/m3_swin3d.yaml $WANDB $RESUME
!cat results/logs/swin3d_t_cls.csv

## C. Đánh giá M3 trên 290 video test + MIL trên feature Swin đã fine-tune

In [ ]:
!python -m src.evaluate --config configs/m3_swin3d.yaml --ckpt weights/swin3d_t_cls.pt --mode cls
!python -m src.extract_features --config configs/m3_swin3d.yaml --ckpt weights/swin3d_t_cls.pt --batch_size 64
!python -m src.train_mil --config configs/m3_swin3d.yaml
!python -m src.benchmark --config configs/m3_swin3d.yaml --ckpt weights/swin3d_t_cls.pt --mode cls || true
!ls results/tables

## D. Đóng gói kết quả để tải về / gửi TV2
Feature (~150 MB/model) để lại trong output nếu cần train MIL lại.

In [ ]:
!zip -qr /kaggle/working/tv3_results.zip results/tables results/logs results/predictions weights/*.pt
!ls -lh /kaggle/working/tv3_results.zip